# Coupled mechanochemical model of nuclear mechanosensing on nanopillars - minimal

Here, we implement the model originally presented in [Sun et al. 2016, Biophysical Journal](https://www.sciencedirect.com/science/article/pii/S0006349516302430?via%3Dihub) and then expanded by [Scott et al. 2021, PNAS](https://www.pnas.org/doi/10.1073/pnas.2021571118). The goal is to describe the nuclear translocation of the mechanosensitive transcriptional regulators YAP/TAZ, given some mechanical input from the substrate. 

Here, we specifically consider cell geometries on substrates with cylindrical nanopillar structures, where the nuclear shape is allowed to evolve over time due to a downward force attributed to the perinuclear actin cap.

The geometry in this model is divided into 4 domains - two volumes and two surfaces:
- plasma membrane (PM)
- Cytosol (Cyto)
- Nuclear envelope (NE)
- Interior of the nucleus (Nuc)

This model has 18 species, including one PM species:
* RhoA_GTP: activated form of RhoA localized to the PM

The cytosolic species include:
* pFAK: phosphorylated FAK (unphosphorylated accounted for by mass conservation)
* RhoA_GDP: inactive form of RhoA
* ROCK_A: activated Rho-associated protein kinase
* mDia_A: activated version of the formin mDia
* Myo_A: activated myosin (associated with stress fiber formation)
* LIMK_A: activated LIM kinase
* Cofilin_NP: nonphosphorylated (activated) cofilin
* FActin: polymerized form of actin
* GActin: monomeric form of actin
* YAPTAZ: dephosphorylated form of YAP/TAZ in the cytosol
* YAPTAZ_seq: phosphorylated form of YAP/TAZ (preferentially remains cytosolic)

The nuclear membrane species include:
* eta: density of nuclear membrane per unit NE surface
* LaminA: dephosphorylated form of Lamin A, a structural component of the nuclear lamina
* LaminAp: phosphorylated form of Lamin A
* NPC_A: activated (open) nuclear pore complexes
* NPC: inactivated (closed) nuclear pore complexes

Finally, the nucleus contains:
* YAPTAZ_nuc: concentration of YAP/TAZ in the nucleus

In [ ]:
import dolfin as d
import sympy as sym
import numpy as np
import pathlib
import logging
import argparse
import matplotlib.pyplot as plt


from smart import config, mesh, model, mesh_tools
from smart.units import unit
from smart.model_assembly import (
    Compartment,
    Parameter,
    Reaction,
    Species,
    sbmodel_from_locals,
)
import sys
from scipy.optimize import fsolve

try:
    import ufl_legacy as ufl
except ImportError:
    import ufl

from mech_parser_args import add_coupled_minimal_arguments
here = pathlib.Path.cwd()
sys.path.insert(0, (here / ".." / "scripts").as_posix())

smart_logger = logging.getLogger("smart")
smart_logger.setLevel(logging.DEBUG)
logger = logging.getLogger("mechanotransduction")
logger.setLevel(logging.INFO)
logger.info("Starting mechanotransduction example")
sys.path.insert(0, "/root/shared/gitrepos/smart-mechanotransduction/mesh-files")
import spread_cell_mesh_generation as mesh_gen

If running a series of tests, we use `argparse` to read in the test conditions. Otherwise, we load default parameters to the `args` dict.

In [ ]:
parser = argparse.ArgumentParser()
add_coupled_minimal_arguments(parser)
try:
    shell = get_ipython().__class__.__name__
    args = {}
    if shell == 'ZMQInteractiveShell': # then running a jupyter notebook
        args["time_step"] = 0.01
        args["e_val"] = 7e7
        args["curv_sens"] = 2
        args["reaction_rate_on_np"] = 1
        args["npc_slope"] = 0.0
        args["a0_npc"] = 5.0
        args["nuc_compression"] = 0.0
        args["WASP_rate"] = 0.01
        cur_dir = str(pathlib.Path.cwd() / "..")
        # args["mesh_folder"] = pathlib.Path(f"{cur_dir}/meshes/nanopillars_nonuc/nanopillars_h1.0_p2.5_r0.25_cellRad17.45")
        # args["mesh_folder"] = pathlib.Path(f"{cur_dir}/meshes/nanopillars_nonuc/nanopillars_h3.0_p3.5_r0.5_cellRad15.5")
        args["mesh_folder"] = pathlib.Path(f"{cur_dir}/meshes/nanopillars_nonuc/nanopillars_h1.0_p1.0_r0.1_cellRad16.55")
        args["outdir"] = pathlib.Path(f"/root/scratch/nuc_mechanics_data/results_testCoupledMech_withYAPTAZcalc")
        args["force_val"] = 500.0
        args["t0_deform"] = 100.0
except:
    args = {}
if len(args) == 0: # then most likely running as a script
    args = vars(parser.parse_args())

timer = d.Timer("mechanotransduction-example")
incl_pore = False
no_substrate = False
laminOnly = False

import re
if "indent" in str(args["mesh_folder"]):
    contactRad = 15.5
    zOffset = 3.0
    nanopillar_height = 3.0
    nanopillar_rad = 0.5
    nanopillar_spacing = 3.5
elif "movenuc" in str(args["mesh_folder"]):
    contactRad = 17.45
    zOffset = 1.0
    nanopillar_height = 1.0
    nanopillar_rad = 0.25
    nanopillar_spacing = 2.5
else:
    contactRad = re.findall(r"cellRad.*", str(args["mesh_folder"]))
    contactRad = float(contactRad[0][7:])
    zOffset = re.findall(r"_h.*_p", str(args["mesh_folder"]))
    zOffset = float(zOffset[0][2:-2])
    nanopillar_height = zOffset
    nanopillar_spacing = re.findall(r"_p.*_r", str(args["mesh_folder"]))
    nanopillar_spacing = float(nanopillar_spacing[0][2:-2])
    nanopillar_rad = re.findall(r"_r.*_c", str(args["mesh_folder"]))
    nanopillar_rad = float(nanopillar_rad[0][2:-2])
# args["e_val"] = (27/128)*(2*nanopillar_rad/nanopillar_height)**3 * args["e_val"]

Now we define units for use in the model.

In [ ]:
# Aliases - base units
uM = unit.uM
um = unit.um
molecule = unit.molecule
sec = unit.sec
dimensionless = unit.dimensionless
# Aliases - units used in model
D_unit = um**2 / sec
flux_unit = uM * um / sec
vol_unit = uM
surf_unit = molecule / um**2
# stiffness units
kPa = unit.kilopascal

# Model generation

For each step of model generation, refer to SMART Example 3 or API documentation for further details.

We first define compartments and the compartment container. Note that we can specify nonadjacency for surfaces in the model, which is not required, but can speed up the solution process.

In [ ]:
# Cyto = Compartment("Cyto", 3, um, 1,
#                    deform=["0","0","0"], manual_update = True)
# Contact = Compartment("Contact", 2, um, 11, deform=["0","0","0"])
# PM = Compartment("PM", 2, um, 10, deform=["0","0","0"], manual_update = True)
Nuc = Compartment("Nuc", 3, um, 2,
                  deform=["0","0","0"], manual_update = True)
R0 = 4.1
z0 = 1.04
NE = Compartment("NE", 2, um, 10,
                 deform=["0","0","0"], manual_update = True,
                 normals = [f"x[0]/{R0}",f"x[1]/{R0}",f"(x[2]-{R0+z0})/{R0}"])
# PM.specify_nonadjacency(["NE", "Nuc"])
# Contact.specify_nonadjacency(["NE", "Nuc"])
# NE.specify_nonadjacency(["Contact", "PM"])

Now we initialize all species in the model.

In [ ]:
if args["a0_npc"] > 0:
    # aNPC = Species("aNPC", 0.0, dimensionless, 0.0, D_unit, "NE")
    aNPC = Parameter.mesh_quantity("aNPC", 0.0, dimensionless, compartment="NE")
init_vals = [0.3, 1.0, 33.6, 0.0, 0.01, 1.5, 0.1, 1.8, 17.9, 482.4, 0.0, 0.0, 0.7, 0.2, 0.7, 0.0]
sp_names = ["pFAK", "RhoA_GDP", "RhoA_GTP", "ROCK_A", "mDia_A", "Myo_A", "LIMK_A", "Cofilin_NP",
            "FActin", "GActin", "LaminA", "NPC_A", "YAPTAZ", "YAPTAZ_seq", "YAPTAZ_nuc"]
init_vec = init_vals

# GActin = Species("GActin", init_vec[9], vol_unit, 13.37, D_unit, "Cyto")
# FActin = Species("FActin", init_vec[8], vol_unit, 0.6, D_unit, "Cyto")
LaminA = Species("LaminA", init_vec[10], surf_unit, 0.001, D_unit, "NE",
                  alt_deform=["0.0","0.0","0.0"], alt_manual_update=True)
if not laminOnly:
    NPC_A = Species("NPC_A", init_vec[11], surf_unit, 0.001, D_unit, "NE")
    # YAPTAZ = Species("YAPTAZ", init_vec[12], vol_unit, 80.0, D_unit, "Cyto")  # non-phosphorylated in the cytosol
    # YAPTAZ_seq = Species("YAPTAZ_seq", init_vec[13], vol_unit, 4.0, D_unit, "Cyto")  # seq in the cytosol
    YAPTAZ_nuc = Species("YAPTAZ_nuc", init_vec[14], vol_unit, 4.0, D_unit, "Nuc")

Now we define reactions and parameters across 3 modules.

Module A: Substrate stiffness -> initial signaling events

Initialize nuclear geometry in cytosol (smallest force that allows it to fit). This can take a long time...

In [ ]:
import nuclear_deformation_mixed3d as nuc_def
nuc_args = dict()
nuc_args["nuc_only"] = True
nuc_args["mesh_folder"] = args["mesh_folder"]
nuc_args["max_force"] = 0.0 # only for initial phase
nuc_args["start_force"] = 0.0
nuc_args["bulk_mod"] = 1e8
nuc_args["nanopillar_radius"] = 0.4 #nanopillar_rad
nuc_args["nanopillar_height"] = 3.0 #nanopillar_height
nuc_args["nanopillar_spacing"] = 3.0 #nanopillar_spacing
nuc_args["outdir"] = args["outdir"] / "mech_data"
nuc_args["contactRad"] = contactRad

# loadInitDef = pathlib.exists(args["mesh_folder"] / "uinit")
# if loadInitDef:
#     test_mesh_file = d.XDMFFile(str(args["mesh_folder"] / "uinit" / "u_np_ellipsoid.xdmf"))
#     test_mesh = d.Mesh()
#     test_mesh_file.read(test_mesh)

#     import h5py
#     test_file_ulin = str(args["mesh_folder"] / "uinit" / "u_np_fullcell.h5")
#     test_file_alin = str(args["mesh_folder"] / "uinit" / "a_np_fullcell.h5")
#     test_file_aInner = str(args["mesh_folder"] / "uinit" / "a_np_inner.h5")
#     test_file_uInner = str(args["mesh_folder"] / "uinit" / "u_np_inner.h5")
#     cur_file = h5py.File(test_file_ulin, 'r')#d.HDF5File(d.MPI.comm_world, test_file_ulin, "r")
#     # cur_array = d.Vector()
#     # cur_file.read(cur_array, f"VisualisationVector/{i}", True)
#     test_idx = 0
#     keep_going = True
#     while keep_going:
#         try:
#             cur_array = cur_file['VisualisationVector'][f'{test_idx}'][()]
#             test_idx += 1
#         except:
#             keep_going = False
    
#     start_idx = test_idx
#     cur_array = cur_array.reshape((3*len(test_mesh.coordinates()),1))
#     cur_file.close()

#     a_file = h5py.File(test_file_alin, 'r')
#     a_array = a_file['VisualisationVector'][f'{start_idx}'][()]
#     a_array = a_array.reshape((3*len(test_mesh.coordinates()),1))
#     a_file.close()

#     aInner_file = h5py.File(test_file_aInner, 'r')
#     a_array = a_file['VisualisationVector'][f'{start_idx}'][()]
#     a_array = a_array.reshape((3*len(test_mesh.coordinates()),1))
#     a_file.close()

#     a_file = h5py.File(test_file_alin, 'r')
#     a_array = a_file['VisualisationVector'][f'{start_idx}'][()]
#     a_array = a_array.reshape((3*len(test_mesh.coordinates()),1))
#     a_file.close()

#     dof_map = d.dof_to_vertex_map(Vtest)[:]
#     # array matches mesh ordering; reorder according to dof mapping for Vcur
#     ulin.vector().set_local(cur_array[dof_map])
#     ulin.vector().apply("insert")
#     alin.vector().set_local(a_array[dof_map])
#     alin.vector().apply("insert")
    
#     kRamp = np.loadtxt(str(args["mesh_folder"] / "uinit" / "kRamp.txt"))
#     pRamp = np.loadtxt(str(args["mesh_folder"] / "uinit" / "pRamp.txt"))
#     kStart = kRamp[start_idx]
#     pStart = pRamp[start_idx]

    

#     nuc_args["u0"] = ...

nuc_dict = nuc_def.start_nuc_mech(nuc_args)

In [ ]:
deformed_mesh_file = args["outdir"] / "mesh" / "nuc_mesh.h5"
uLagrange = nuc_dict["uLagrange"]
nuc_mesh = uLagrange.function_space().mesh()
mf3 = nuc_dict["mf3"]
mf2 = nuc_dict["mf2"]
mf3.array()[mf3.array()==1] = 2
mesh_tools.write_mesh(nuc_mesh, mf2, mf3, deformed_mesh_file)
# deformed_mesh_file = pathlib.Path(nuc_args["mesh_folder"]) / "deformed" / "spreadCell_mesh.h5"
# deformed_mesh_curv = pathlib.Path(nuc_args["mesh_folder"]) / "deformed" / "spreadCell_mesh.h5"
parent_mesh = mesh.ParentMesh(
    mesh_filename=str(deformed_mesh_file),
    mesh_filetype="hdf5",
    name="parent_mesh",
    # curvature=pathlib.Path(nuc_args["mesh_folder"]) / "deformed" / "curvatures.xdmf",
    # extra_keys=["subdomain0_2"],
    # key_for_facets="subdomain0_2"
)
mf2 = d.MeshFunction("size_t", parent_mesh.dolfin_mesh, 2, value=0)
hdf5 = d.HDF5File(parent_mesh.dolfin_mesh.mpi_comm(), parent_mesh.mesh_filename, "r")
hdf5.read(mf2, f"/mf2")
hdf5.close()
# assert mf2.mesh().id() == parent_mesh.mf["facets"].mesh().id(),\
#     "Mesh for subdomain does not match"
# # set substrate facets to 11
# parent_mesh.mf["facets"].array()[parent_mesh.mf["facets"].array() > 0] = 11
# # assign PM markers from original mesh
# zero_idx = parent_mesh.mf["facets"].array() == 0
# parent_mesh.mf["facets"].array()[zero_idx] = mf2.array()[zero_idx]

# load cell markers
mf3 = d.MeshFunction("size_t", parent_mesh.dolfin_mesh, 3, value=0)
hdf5 = d.HDF5File(parent_mesh.dolfin_mesh.mpi_comm(), parent_mesh.mesh_filename, "r")
hdf5.read(mf3, f"/mf3")
hdf5.close()

In [ ]:
mesh_ne = d.create_meshview(mf2, 10)
dx_nm = d.Measure("dx", mesh_ne)
ref_int =  45.74295 * 2 #d.assemble_mixed(1.0*dx_nm)
proj_NE = d.VectorFunctionSpace(mesh_ne, "P", 1)
u_NE = d.interpolate(uLagrange, proj_NE)
F_NE = d.Identity(u_NE.geometric_dimension()) + d.grad(u_NE)
npc_factor = ref_int / d.assemble_mixed(d.det(F_NE)*dx_nm)
lamin_factor = ref_int / d.assemble_mixed(d.det(F_NE)*dx_nm)
Fcur = d.Identity(uLagrange.geometric_dimension()) + d.grad(uLagrange)
vol_nuc = d.assemble_mixed(d.det(Fcur)*d.Measure("dx", uLagrange.function_space().mesh()))
vol_cyto = 437.0 * 2 # multiplied by 2 from prev geo because going from 1/8 to 1/4 geo

In [ ]:
FActin0 = 90.0
FActin = Parameter.mesh_quantity("FActin", FActin0, uM, compartment="NE")

Module C: nucleo-cytoplasmic transport

In [ ]:
# c3a: LaminA dephos. and phos.
LaminA_tot = Parameter("LaminA_tot", lamin_factor*3500.0, surf_unit)
LaminAp = Species("LaminAp", lamin_factor*3500.0/2, surf_unit, 0.01, D_unit, "NE", 
                   alt_deform=["0.0","0.0","0.0"], alt_manual_update=True)
LaminA.initial_condition = lamin_factor*3500.0/2
k_fl = Parameter("k_fl", 0.46, 1 / sec)
p = Parameter("p", 9.0e-6, kPa / uM**2.6)
C_LaminA = Parameter("C_LaminA", 100.0, kPa)
k_rl = Parameter("k_rl", 0.001, 1 / sec)
c3a = Reaction(
    "c3a",
    ["LaminAp"],
    ["LaminA"],
    param_map={
        "k_fl": "k_fl",
        "p": "p",
        "C_LaminA": "C_LaminA",
        "k_rl": "k_rl",
        "FActin": "FActin",
    },
    explicit_restriction_to_domain="NE",
    eqn_f_str="LaminAp*k_fl*p*FActin**2.6/(C_LaminA + p*FActin**2.6) - k_rl*LaminA",
)

# c3b: LaminA disassembly
kdeg_lamin = Parameter("kdeg_lamin", 0.001, 1/sec)
c3b = Reaction(
    "c3b",
    ["LaminAp"],
    [],
    param_map={
        "kdeg": "kdeg_lamin",
    },
    explicit_restriction_to_domain="NE",
    eqn_f_str="kdeg*LaminAp",
)

# c3c: LaminA reassembly
kassembly_lamin = Parameter("kassembly_lamin", 0.001, 1/sec)
LaminA_set = Parameter("LaminA_set", 3500.0*lamin_factor, surf_unit)
c3c = Reaction(
    "c3c",
    [],
    ["LaminA"],
    param_map={
        "kassembly": "kassembly_lamin",
        "LaminA_set": "LaminA_set",
    },
    explicit_restriction_to_domain="NE",
    eqn_f_str="kassembly*(LaminA_set - LaminA)",
)

if laminOnly:
    kdeg_yap = Parameter("kdeg_yap", 0.0, 1/sec)
    YAPTAZ_nuc = Species("YAPTAZ_nuc", 0.7, uM, 4.0, D_unit, "Nuc")
    rYAP = Reaction("rYAP", ["YAPTAZ_nuc"], [], 
                    param_map={"kdeg":"kdeg_yap"},
                    eqn_f_str="YAPTAZ_nuc*kdeg")
else:
    # Module C: nucleo-cytoplasmic transport
    # c1: YAP/TAZ deseq. and sequestration in the cytosol
    # k_CN = Parameter("k_CN", 0.56, 1 / sec)
    # k_CY = Parameter("k_CY", 0.00076, 1 / (sec * uM**2))
    # k_NC = Parameter("k_NC", 0.14, 1 / sec)
    Myo_A = Parameter("Myo_A", 4.0, uM)
    # c1 = Reaction(
    #     "c1",
    #     ["YAPTAZ_seq"],
    #     ["YAPTAZ"],
    #     param_map={"k_CN": "k_CN", "k_CY": "k_CY", 
    #                 "k_NC": "k_NC", "Myo_A": "Myo_A", "FActin": "FActin"},
    #     species_map={
    #         "YAPTAZ": "YAPTAZ",
    #         "YAPTAZ_seq": "YAPTAZ_seq",
    #     },
    #     eqn_f_str="YAPTAZ_seq*(k_CN + k_CY*FActin*Myo_A) - k_NC*YAPTAZ",
    # )

    # c2: nuclear membrane transport
    # eta0val = 3.0
    # eta = Species("eta", eta0val, dimensionless, 0.1, D_unit, "NE")
    # kcycle_nm = Parameter("kcycle_nm", 0.01, 1/sec)
    # eta0 = Parameter("eta0", eta0val, dimensionless)
    # c2 = Reaction("c2", [], ["eta"],
    #                 param_map={"kcycle_nm": "kcycle_nm", "eta0": "eta0"},
    #     explicit_restriction_to_domain="NE",
    #     eqn_f_str="kcycle_nm*(eta0-eta)")
    # c4: opening and closing of NPCs
    # eta0val = 3
    # eta = Parameter("eta", 3.0, dimensionless)
    NPCtot = 6.5*npc_factor
    k_fNPCval = 2.8e-7
    k_rNPCval = 8.7
    K_NPC = k_fNPCval / k_rNPCval
    NPC_A.initial_condition = NPCtot * ((K_NPC*LaminA.initial_condition*FActin0*Myo_A.value)/
                                        (1 + K_NPC*LaminA.initial_condition*FActin0*Myo_A.value))
    NPC = Species("NPC", NPCtot - NPC_A.initial_condition, surf_unit, 0.001, D_unit, "NE")
    # NPC_tot = Parameter.from_expression("NPC_tot", f"(6.5/{eta0val})*{npc_factor}*(1 - {args['npc_slope']}*(z-{zMin})/({zMax-zMin}))", surf_unit)
    k_fNPC = Parameter("k_fNPC", k_fNPCval, 1 / (sec * uM**2 * surf_unit))
    k_rNPC = Parameter("k_rNPC", k_rNPCval, 1 / sec)
    c4 = Reaction(
        "c4",
        ["NPC"],
        ["NPC_A"],
        param_map={"k_fNPC": "k_fNPC", "k_rNPC": "k_rNPC", 
                "Myo_A": "Myo_A", "FActin": "FActin"},
        explicit_restriction_to_domain="NE",
        species_map={
            "LaminA": "LaminA",
        },
        eqn_f_str="NPC*k_fNPC*LaminA*FActin*Myo_A - k_rNPC*NPC_A",
    )

    # c5: nuclear translocation of YAP/TAZ
    Nconv = 602.2
    Ytot = Nconv*(0.9*vol_cyto + 0.7*vol_nuc)
    K_CN = 4.0
    K_CY = 5.429e-3
    k_insoloval = 1.0
    k_in2val = 10.0
    k_outval = 1.0
    K_solo = k_insoloval/k_outval
    K_2 = k_in2val/k_outval
    phiNP = (K_CN + K_CY*FActin0*Myo_A.value)/(1+ K_CN + K_CY*FActin0*Myo_A.value)
    Ynuc0 = (Ytot*phiNP/Nconv) * ((K_solo + K_2*NPC_A.initial_condition)/
                                  ((K_solo + K_2*NPC_A.initial_condition)*phiNP*vol_nuc + vol_cyto))
    YAPTAZ_nuc.initial_condition = Ynuc0
    Ycyto = (Ytot - Ynuc0*vol_nuc*Nconv)/(vol_cyto*Nconv)
    Yfree = Ycyto*phiNP
    k_insolo = Parameter("k_insolo", k_insoloval, surf_unit / (sec * uM))
    k_in2 = Parameter("k_in2", k_in2val, 1 / (sec * uM))
    k_out = Parameter("k_out", k_outval, surf_unit / (sec * uM))
    YAPTAZ = Parameter("YAPTAZ", Yfree, uM)
    if args["a0_npc"] > 0:
        a0_NPC = Parameter("a0_NPC", args["a0_npc"], dimensionless)
        c5 = Reaction(
            "c5",
            [],
            ["YAPTAZ_nuc"],
            param_map={"k_insolo": "k_insolo", "k_in2": "k_in2", "k_out": "k_out", 
                       "a0_NPC": "a0_NPC", "aNPC": "aNPC", "YAPTAZ": "YAPTAZ"},
            species_map={"YAPTAZ_nuc": "YAPTAZ_nuc", "NPC_A": "NPC_A"},
            explicit_restriction_to_domain="NE",
            eqn_f_str="YAPTAZ*exp((aNPC-1)/a0_NPC)*(k_insolo + k_in2*NPC_A) - k_out*YAPTAZ_nuc",
        )
    else:
        c5 = Reaction(
            "c5",
            [],
            ["YAPTAZ_nuc"],
            param_map={"k_insolo": "k_insolo", "k_in2": "k_in2", "k_out": "k_out", "YAPTAZ": "YAPTAZ"},
            species_map={"YAPTAZ_nuc": "YAPTAZ_nuc", "NPC_A": "NPC_A"},
            explicit_restriction_to_domain="NE",
            eqn_f_str="YAPTAZ*(k_insolo + k_in2*NPC_A) - k_out*YAPTAZ_nuc",
        )

    # # c7: nuclear rupture events
    if incl_pore:
        pore = Parameter.from_expression("pore", 
                                        f"(1-exp(-t/1.0))*exp(-(pow(x,2)+pow(y,2)+pow(z-{zCenter},2))/5.0)", 
                                        dimensionless)
        # c7 and c8: YAP leak due to lysis
        k_inPore = Parameter("k_inPore", 100.0, surf_unit / (sec * uM))
        k_outPore = Parameter("k_outPore", 10.0, surf_unit / (sec * uM))
        c7 = Reaction(
            "c7",
            [],
            ["YAPTAZ_nuc"],
            param_map={"k_inPore": "k_inPore", "k_outPore": "k_outPore", "pore": "pore", "YAPTAZ": "YAPTAZ"},
            species_map={"YAPTAZ_nuc": "YAPTAZ_nuc"},
            explicit_restriction_to_domain="NE",
            eqn_f_str="YAPTAZ*pore*k_inPore - k_outPore*pore*YAPTAZ_nuc",
        )

In [ ]:
pc, sc, cc, rc = sbmodel_from_locals(locals().values())

In [ ]:
# set config for current run
configCur = config.Config()
configCur.flags.update(
    {
        "allow_unused_components": True,
    }
)
configCur.solver.update(
    {
        "final_t": 10000.0,
        "initial_dt": args["time_step"],
        "time_precision": 6,
        "use_snes": True,
        "attempt_timestep_restart_on_divergence": True,
        "reset_timestep_for_negative_solution": False,
    }
)

In [ ]:
model_cur = model.Model(pc, sc, cc, rc, configCur, parent_mesh)
model_cur.initialize(initialize_solver=True)
# Write initial condition(s) to file
results = dict()
result_folder = args["outdir"]
result_folder.mkdir(exist_ok=True, parents=True)

Initialize deformation fields.

In [ ]:
uglobal = uLagrange #nuc_dict["uLagrangeFull"]
# proj_cyto = cc["Cyto"].deform_func.function_space()
proj_NE = cc["NE"].deform_func.function_space()
proj_Nuc = cc["Nuc"].deform_func.function_space()
# cc["Cyto"].deform_func.assign(d.interpolate(uglobal, proj_cyto))
# cc["Cyto"].deform_prev.assign(d.interpolate(uglobal, proj_cyto))
cc["NE"].deform_func.assign(d.interpolate(uglobal, proj_NE))
cc["NE"].deform_prev.assign(d.interpolate(uglobal, proj_NE))
cc["Nuc"].deform_func.assign(d.interpolate(uglobal, proj_Nuc))
cc["Nuc"].deform_prev.assign(d.interpolate(uglobal, proj_Nuc))

# map inner velocities
uInnerVals = sc["LaminA"].alt_deform_func.vector()[:] #nuc_dict["uLagrangeInner"]
NECoords = sc["LaminA"].alt_deform_func.function_space().tabulate_dof_coordinates()
nucRad1 = nuc_dict["nucRad1"]
nucRad2 = nuc_dict["nucRad2"]
rthickness = nuc_dict["rthickness"]
zthickness = nuc_dict["zthickness"]
z0 = min(NECoords[:,2])
for i in range(0,len(NECoords),3):
    xcur = NECoords[i]
    rcur = np.sqrt(xcur[0]**2 + xcur[1]**2 + (xcur[2]-nucRad2-z0)**2)
    if np.abs(rcur - nucRad1) > 0.01*nucRad1:
        raise ValueError("Not at the right point")
    if nucRad1 != nucRad2 or rthickness != zthickness:
        raise ValueError("inner extrapolation does not work for this geo")
    xcur_inner = xcur * (nucRad2-rthickness)/nucRad2
    xcur_inner[2] = nucRad2 + z0 + (xcur[2]-nucRad2-z0) * (nucRad2-rthickness)/nucRad2
    uInnerVals[i:i+3] = (xcur_inner - xcur) + uglobal(xcur_inner)

for inner_sp in ["LaminA", "LaminAp"]:
    sc[inner_sp].alt_deform_func.vector().set_local(uInnerVals)
    sc[inner_sp].alt_deform_func.vector().apply("insert")
    sc[inner_sp].alt_deform_prev.vector().set_local(uInnerVals)
    sc[inner_sp].alt_deform_prev.vector().apply("insert")

In [ ]:
# # b7: curvature-dependent actin polymerization at the PM
actinBoost = 20.0
actinDecay = 0.2
npSpacing = nuc_args["nanopillar_spacing"]
npRad = nuc_args["nanopillar_radius"]
hNP = nuc_args["nanopillar_height"]
xMax = np.ceil(2*nucRad1 / npSpacing) * npSpacing
xNP = np.arange(-xMax, xMax+1e-12, npSpacing)
yNP = np.arange(-xMax, xMax+1e-12, npSpacing)
xNP, yNP = np.meshgrid(xNP, yNP)
xNP = xNP.flatten()
yNP = yNP.flatten()
FActin_func = pc["FActin"].dolfin_function
FActin_coords = FActin_func.function_space().tabulate_dof_coordinates()
Fvals = FActin_func.vector()[:]
dSteric = 0.2
for i in range(len(FActin_coords)):
    xDef = FActin_coords[i] + uglobal(FActin_coords[i])
    all_dist = np.sqrt((xNP-xDef[0])**2 + (yNP-xDef[1])**2)
    if np.any(all_dist <= npRad) or (len(xNP)==0):
        curDist = xDef[2]+dSteric
        PMcurv = 0.0
    else:
        min_idx = np.argmin(all_dist)
        if xDef[2] > 0.0:
            curDist = np.sqrt((all_dist[min_idx]-npRad)**2 + xDef[2]**2)
        else:
            curDist = all_dist[min_idx] - npRad
        PMcurv = 1/(2*(npRad+0.05))
    Fvals[i] = FActin0 + actinBoost*np.exp(-curDist/actinDecay)*np.exp(PMcurv/2.0)
FActin_func.vector().set_local(Fvals)
FActin_func.vector().apply("insert")
FActin_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"FActin.xdmf"))
FActin_file.parameters["flush_output"] = True
FActin_file.write(FActin_func, model_cur.t)

Initialize values of nuclear stretch (alpha).

In [ ]:
if args["a0_npc"] > 0:
    avec = nuc_dict["a_vector"]
    # afun = ufl.sqrt(avec[0]**2 + avec[1]**2 + avec[2]**2)
    # amesh = avec.function_space().mesh()
    # Vscalar_a = d.FunctionSpace(avec.function_space().mesh(), "P", 1)
    # assert np.all(d.vertex_to_dof_map(sc["aNPC"].V) == d.vertex_to_dof_map(Vscalar_a))
    # assert np.all(d.vertex_to_dof_map(pc["aNPC"].dolfin_function.function_space()) == d.vertex_to_dof_map(Vscalar_a))
    # aproj = d.project(afun, Vscalar_a)
    # avals = sc["aNPC"].u["u"].vector()[:]
    # avals[sc["aNPC"].dof_map] = aproj.vector()[:]
    # for tag in ["u", "n"]:
    #     sc["aNPC"].u[tag].vector().set_local(avals)
    #     sc["aNPC"].u[tag].vector().apply("insert")
    afunc = pc["aNPC"].dolfin_function
    avals = afunc.vector()[:]
    acoords = afunc.function_space().tabulate_dof_coordinates()
    for i in range(len(acoords)):
        avec_cur = avec(acoords[i])
        avals[i] = np.sqrt(avec_cur[0]**2 + avec_cur[1]**2 + avec_cur[2]**2)
    ainit = np.mean(avals)
    avals /= ainit
    pc["aNPC"].dolfin_function.vector().set_local(avals)
    pc["aNPC"].dolfin_function.vector().apply("insert")
    aNPC_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"aNPC.xdmf"))
    aNPC_file.parameters["flush_output"] = True
    aNPC_file.write(pc["aNPC"].dolfin_function, model_cur.t)

Write out settings used in current simulation.

In [ ]:
if model_cur.mpi_comm_world.rank == 0:
    import json
    # Dump config to results folder
    (result_folder / "config.json").write_text(
        json.dumps(
            {
                "solver": configCur.solver.__dict__,
                "flags": configCur.flags.__dict__,
                "reaction_database": configCur.reaction_database,
                "mesh_file": str(args["mesh_folder"]),
                "outdir": str(args["outdir"]),
                "time_step": args["time_step"],
            }
        )
    )

Initialize XDMF files for storing output.

In [ ]:
for species_name, species in model_cur.sc.items:
    results[species_name] = d.XDMFFile(
        model_cur.mpi_comm_world, str(result_folder / f"{species_name}.xdmf")
    )
    results[species_name].parameters["flush_output"] = True
    results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)

Jdict = dict()
for c in model_cur._active_compartments:
    cname = c.name
    results[cname] = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"{cname}_udef.xdmf"))
    results[cname].parameters["flush_output"] = True
    udef = c.deform_func
    results[cname].write(udef, model_cur.t)
    jname = f"J_{c.name}"
    results[jname] = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"{jname}.xdmf"))
    results[jname].parameters["flush_output"] = True
    Fcur = d.Identity(udef.geometric_dimension()) + d.grad(udef)
    Vscalar = d.FunctionSpace(c.dolfin_mesh, "P", 1)
    Jdict[cname] = d.project(d.det(Fcur), Vscalar)
    results[jname].write(Jdict[cname], model_cur.t)
if sc["LaminA"].alt_deform_func != None:
    c = sc["LaminA"].compartment
    cnamealt = f"{c.name}_alt"
    results[cnamealt] = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"{cnamealt}_udef.xdmf"))
    results[cnamealt].parameters["flush_output"] = True
    udef = sc["LaminA"].alt_deform_func
    results[cnamealt].write(udef, model_cur.t)

In [ ]:
# Set loglevel to warning in order not to pollute notebook output
# smart_logger.setLevel(logging.INFO)

# define integration measures
# nucMesh = model_cur.cc["Nuc"].dolfin_mesh
# dxNuc = d.Measure("dx", domain=nucMesh)
# int_val = d.assemble_mixed(model_cur.sc["YAPTAZ_nuc"].u["u"] * dxNuc)
# vol_nuc = d.assemble_mixed(1.0 * dxNuc)
# YAPTAZ_nuc_vec = np.array([int_val / vol_nuc])

Solve system over all timesteps. 

In [ ]:
# Solve
force0 = nuc_dict["kRamp"][-1]
# nuc_dict["kRamp"][-1] = force0 # to avoid initial jump in force
forceMax = args["force_val"]
# Vmarker = d.FunctionSpace(cc["Cyto"].dolfin_mesh, "P", 1)
# cytosolic_marker = d.interpolate(d.Expression("1.0",degree=1), Vmarker)
# marker_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"cytosolic_marker.xdmf"))
# marker_file.parameters["flush_output"] = True
# marker_file.write(cytosolic_marker, model_cur.t)
# eta_func = model_cur.sc["eta"].u["u"]
t0_deform = args["t0_deform"]
stress_var = nuc_dict["surf_stress"]
stress_var.set_allow_extrapolation(True)
V_NE = d.FunctionSpace(cc["NE"].dolfin_mesh, "P", 1)
rupture_marker = d.Function(V_NE) #d.project(1/(sc["LaminA"].sol + sc["LaminAp"].sol), V_NE)
rupture_vec = rupture_marker.vector()[:]
NEcoords = V_NE.tabulate_dof_coordinates()
for i in range(len(NEcoords)):
    stressCur = stress_var(NEcoords[i])
    rupture_vec[i] = (np.sqrt(stressCur[0]**2 + stressCur[1]**2 + stressCur[2]**2))# /
                    #   sc["LaminA"].sol.vector()[i] + sc["LaminAp"].sol.vector()[i])
rupture_marker.vector().set_local(rupture_vec)
rupture_marker.vector().apply("insert")
rupture_file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"rupture_marker.xdmf"))
rupture_file.parameters["flush_output"] = True
rupture_file.write(rupture_marker, model_cur.t)
while True:
    # for sp in ["NPC", "NPC_A"]:
    #     Dcur = d.project(d.Constant(0.003)/eta_func, sc[sp].V)
    #     sc[sp].D_dolfin.assign(Dcur)
    logger.info("First, update deformation")
    # consider time step restriction due to force jump
    nuc_dict["tvec"] = model_cur.tvec.copy()
    nuc_dict["tvec"].append(model_cur.t+model_cur.dt)
    forceCur = force0 + (forceMax-force0)*(
        1-np.exp(-float(model_cur.t+model_cur.dt)/t0_deform))
    if forceCur - nuc_dict["kRamp"][-2] > 10.0:
        logger.info("Restricting time step due to force jump")
        forceCur = nuc_dict["kRamp"][-2] + 10.0
        tval = -t0_deform*np.log(((forceMax-force0) - (forceCur-force0))/(forceMax-force0))
        model_cur.set_dt(tval - float(model_cur.t))
    if forceCur - nuc_dict["kRamp"][-2] < 0.02 or forceMax == 0.0:
        logger.info("Skipping mechnical calculation because force has not changed much")
    else:
        nuc_dict["kRamp"][-1] = forceCur
        nuc_dict = nuc_def.solve_next_nuc_step(nuc_dict)
        uglobal = uLagrange #nuc_dict["uLagrangeFull"]
        # cc["Cyto"].deform_func.assign(d.interpolate(uglobal, proj_cyto))
        cc["NE"].deform_func.assign(d.interpolate(uglobal, proj_NE))
        cc["Nuc"].deform_func.assign(d.interpolate(uglobal, proj_Nuc))
        # map inner velocities
        uInnerVals = sc["LaminA"].alt_deform_func.vector()[:]
        for i in range(0,len(NECoords),3):
            xcur = NECoords[i]
            rcur = np.sqrt(xcur[0]**2 + xcur[1]**2 + (xcur[2]-nucRad2-z0)**2)
            xcur_inner = xcur * (nucRad2-rthickness)/nucRad2
            xcur_inner[2] = nucRad2 + z0 + (xcur[2]-nucRad2-z0) * (nucRad2-rthickness)/nucRad2
            uInnerVals[i:i+3] = (xcur_inner - xcur) + uglobal(xcur_inner)
        for inner_sp in ["LaminA", "LaminAp"]:
            sc[inner_sp].alt_deform_func.vector().set_local(uInnerVals)
            sc[inner_sp].alt_deform_func.vector().apply("insert")
            sc[inner_sp].alt_deform_prev.vector().set_local(uInnerVals)
            sc[inner_sp].alt_deform_prev.vector().apply("insert")
        for i in range(len(FActin_coords)):
            xDef = FActin_coords[i] + uglobal(FActin_coords[i])
            all_dist = np.sqrt((xNP-xDef[0])**2 + (yNP-xDef[1])**2)
            if np.any(all_dist <= npRad) or (len(xNP)==0):
                curDist = xDef[2]+dSteric
                PMcurv = 0.0
            else:
                min_idx = np.argmin(all_dist)
                if xDef[2] > 0.0:
                    curDist = np.sqrt((all_dist[min_idx]-npRad)**2 + xDef[2]**2)
                else:
                    curDist = all_dist[min_idx] - npRad
                PMcurv = 1/(2*(npRad+0.05))
            Fvals[i] = FActin0 + actinBoost*np.exp(-curDist/actinDecay)*np.exp(PMcurv/2.0)
        FActin_func.vector().set_local(Fvals)
        FActin_func.vector().apply("insert")
        FActin_file.write(FActin_func, model_cur.t)
        if args["a0_npc"] > 0:
            avec = nuc_dict["a_vector"]
            for i in range(len(acoords)):
                avec_cur = avec(acoords[i])
                avals[i] = np.sqrt(avec_cur[0]**2 + avec_cur[1]**2 + avec_cur[2]**2)/ainit
            pc["aNPC"].dolfin_function.vector().set_local(avals)
            pc["aNPC"].dolfin_function.vector().apply("insert")
            aNPC_file.write(pc["aNPC"].dolfin_function, model_cur.t)
        stress_var = nuc_dict["surf_stress"]
        for i in range(len(NEcoords)):
            stressCur = stress_var(NEcoords[i])
            rupture_vec[i] = (np.sqrt(stressCur[0]**2 + stressCur[1]**2 + stressCur[2]**2) /
                            sc["LaminA"].sol.vector()[i] + sc["LaminAp"].sol.vector()[i])
        rupture_marker.vector().set_local(rupture_vec)
        rupture_marker.vector().apply("insert")
        rupture_file.write(rupture_marker, model_cur.t)

    for c in model_cur._active_compartments:
        cname = c.name
        jname = f"J_{c.name}"
        udef = c.deform_func
        results[cname].write(udef, model_cur.t+model_cur.dt)
        Fcur = d.Identity(udef.geometric_dimension()) + d.grad(udef)
        Vscalar = d.FunctionSpace(c.dolfin_mesh, "P", 1)
        Jdict[cname].assign(d.project(d.det(Fcur), Vscalar))
        results[jname].write(Jdict[cname], model_cur.t+model_cur.dt)
    if sc["LaminA"].alt_deform_func != None:
        c = sc["LaminA"].compartment
        cnamealt = f"{c.name}_alt"
        udef = sc["LaminA"].alt_deform_func
        results[cnamealt].write(udef, model_cur.t)
    # J_file.write(Jproj, model_cur.t+model_cur.dt)

    # udef = cc["Cyto"].deform_func
    # Fcur = d.Identity(udef.geometric_dimension()) + d.grad(udef)
    # Jcur = d.det(Fcur)
    # uprev = cc["Cyto"].deform_prev
    # Fprev = d.Identity(uprev.geometric_dimension()) + d.grad(uprev)
    # Jprev = d.det(Fprev)
    # Vscalar = d.FunctionSpace(cc["Cyto"].dolfin_mesh, "P", 1)
    # cytosolic_marker.assign(d.project(cytosolic_marker*Jprev/Jcur, Vscalar))
    # marker_file.write(cytosolic_marker, model_cur.t+model_cur.dt)
    # for sp in cc["Cyto"].species.values():
    #     u_advected = d.project(sp.u["u"]*Jprev/Jcur, Vscalar)
    #     sp.u["u"].assign(u_advected)

    logger.info(f"Solve for time step {model_cur.t}")
    model_cur.monolithic_solve()
    model_cur.adjust_dt()

    # update values of YAP/TAZ
    Fnuc = d.Identity(cc["Nuc"].deform_func.geometric_dimension()) + d.grad(cc["Nuc"].deform_func)
    phiNP = (K_CN + K_CY*FActin0*Myo_A.value)/(1+ K_CN + K_CY*FActin0*Myo_A.value)
    dxNuc = d.Measure("dx", domain=cc["Nuc"].dolfin_mesh)
    Ycyto = (Ytot - d.assemble_mixed(d.det(Fnuc)*sc["YAPTAZ_nuc"].u["u"]*dxNuc)*Nconv)/(vol_cyto*Nconv)
    pc["YAPTAZ"].value = Ycyto*phiNP
    pc["YAPTAZ"].value_vector = np.vstack((pc["YAPTAZ"].value_vector, [float(model_cur.t), pc["YAPTAZ"].value]))
    pc["YAPTAZ"].dolfin_constant.assign(pc["YAPTAZ"].value)
    YAPTAZ.dolfin_constant.assign(Ycyto*phiNP)
    np.savetxt(str(result_folder / "YAPTAZ.txt"), pc["YAPTAZ"].value_vector)

    # int_val = d.assemble_mixed(Jdict["Nuc"]*model_cur.sc["YAPTAZ_nuc"].u["u"] * dxNuc)
    # vol_nuc = d.assemble_mixed(Jdict["Nuc"] * dxNuc)
    # current_YAPTAZ_nuc = np.array([int_val / vol_nuc])
    # YAPTAZ_nuc_vec = np.concatenate((YAPTAZ_nuc_vec, current_YAPTAZ_nuc))

    # Save results for post processing
    for species_name, species in model_cur.sc.items:
        results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)
    
    np.savetxt(result_folder / "tvec.txt", np.array(model_cur.tvec).astype(np.float32))

    # if model_cur.mpi_comm_world.rank == 0:
    #     np.savetxt(result_folder / "YAPTAZ_nuc.txt", YAPTAZ_nuc_vec.astype(np.float32))
    # End if we've passed the final time
    if model_cur.t >= model_cur.final_t:
        break